# Bambara Speech AI - Google Colab Training Notebook

This notebook is designed to prepare and run training for Bambara speech workloads, including:
- TTS (Text-to-Speech)
- ASR (Automatic Speech Recognition)
- dataset manifest creation
- preprocessing
- inference

Important:
- This is a starter notebook that expects a real Bambara dataset with audio + transcripts.
- For real training, use a GPU-enabled runtime or Colab Pro/Pro+ if available.
- Replace the sample data paths with your own dataset before training.


In [ ]:
# 1) Install dependencies
!python -V
!pip install -q --upgrade pip
!pip install -q numpy pandas soundfile librosa scipy matplotlib tqdm PyYAML
!pip install -q torch torchaudio transformers accelerate datasets sentencepiece scikit-learn
!pip install -q git+https://github.com/huggingface/transformers.git
print('Dependencies installed')


In [ ]:
# 2) Mount Google Drive (optional but recommended)
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')


In [ ]:
# 3) Clone or open project repository
import os
repo_path = '/content/bambara-tts-waxal-clean'
if not os.path.exists(repo_path):
    !git clone https://github.com/adamacoulibalyad670-droid/bambara-tts-waxal-clean.git {repo_path}
else:
    print('Repository already exists')

%cd {repo_path}
!ls -la


In [ ]:
# 4) Create required folders for dataset
!mkdir -p data/audio/train data/audio/val data/audio/test data/transcripts
!mkdir -p data/processed models/tts models/asr outputs
!find data -maxdepth 3 -type d -print


## Upload your dataset

You should place your Bambara audio files in the folders above and create transcript files like this:

```text
data/transcripts/train.txt
train/001.wav Aw ka se ka baara duman min ye?
train/002.wav Mali la denw ka caya.
```

For validation and testing, create similar files in `data/transcripts/val.txt` and `data/transcripts/test.txt`.

You can upload files using the Colab Files panel or by copying from Google Drive.


In [ ]:
# 5) Build manifest CSV files if transcripts are available
!python scripts/build_manifest.py --audio_dir data/audio/train --transcript_file data/transcripts/train.txt --output data/train_manifest.csv
!python scripts/build_manifest.py --audio_dir data/audio/val --transcript_file data/transcripts/val.txt --output data/val_manifest.csv
!python scripts/build_manifest.py --audio_dir data/audio/test --transcript_file data/transcripts/test.txt --output data/test_manifest.csv
!ls -l data
!head -n 5 data/train_manifest.csv


In [ ]:
# 6) Preprocess the dataset
!python -m src.preprocess --manifest data/train_manifest.csv --output_dir data/processed
!ls -l data/processed


## Training for TTS (Text-to-Speech)

Choose one of the recommended backends:
- XTTS-v2
- Coqui TTS
- VITS
- FastSpeech2

This notebook uses the repository starter scripts, which are ready to be replaced with a real training backend when your data is prepared.


In [ ]:
# 7) Train TTS
!python -m src.train_tts --config configs/tts_config.yaml --manifest data/train_manifest.csv


## Training for ASR (Speech-to-Text)

Choose one of the recommended backends:
- Whisper fine-tuning
- Wav2Vec2
- MMS
- HuBERT

This notebook is ready to plug in a real ASR backbone once your Bambara audio and transcripts are available.


In [ ]:
# 8) Train ASR
!python -m src.train_asr --config configs/asr_config.yaml --manifest data/train_manifest.csv


In [ ]:
# 9) Inference examples
!python -m src.infer_tts --text "Aw ka se ka baara duman min ye?"
!python -m src.infer_asr --audio_path data/audio/test/001.wav


## Notes

This notebook is a starter implementation for Bambara speech AI. It prepares the workflow and project structure, but real model performance depends on:
- dataset quality
- transcript correctness
- adequate GPU memory
- careful preprocessing and validation
- use of a production-grade TTS/ASR backbone
